# ERA5 Sea Surface Temperature — Mediterranean Sea
* Use `pystac_client` to search the ERA5-PDS collection on Planetary Computer
* Open the Zarr store directly via `xarray`
* Subset to the Mediterranean domain and compute a monthly mean
* Visualise with `hvplot` (same approach as the Sentinel-2 notebook)

In [ ]:
import pystac_client
import planetary_computer
import xarray as xr
import hvplot.xarray
import numpy as np

# Connect to the Planetary Computer STAC API
catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)

In [ ]:
# Search for the ERA5 analysis item for a specific month
# 'an' (analysis) items contain sea_surface_temperature; 'fc' (forecast) items do not
YEAR_MONTH = "2020-08"

search = catalog.search(collections=["era5-pds"], datetime=YEAR_MONTH)
items = search.item_collection()
print(f"Found {len(items)} items for {YEAR_MONTH}")
for item in items:
    print(f"  {item.id}  (kind={item.properties['era5:kind']})")

In [ ]:
# Select the analysis item and sign it to get a time-limited access credential
an_item = next(i for i in items if i.properties["era5:kind"] == "an")
signed_item = planetary_computer.sign(an_item)

# Grab the sea_surface_temperature asset and its recommended open kwargs
asset = signed_item.assets["sea_surface_temperature"]
open_kwargs = asset.extra_fields["xarray:open_kwargs"].copy()
storage_options = open_kwargs.pop("storage_options")
open_kwargs.pop("engine", None)  # engine kwarg is not accepted by open_zarr

print("Opening:", asset.href)
ds = xr.open_zarr(asset.href, storage_options=storage_options, **open_kwargs)
ds

In [ ]:
# Mediterranean bounding box
# Latitude:  30 N – 47 N
# Longitude: ERA5 uses 0–360, so the Med spans both -6°→42° E
#            which maps to 354–360 (western part) and 0–42 (eastern part)
LAT_MIN, LAT_MAX = 30.0, 47.0
LON_MIN_W, LON_MAX_W = 354.0, 360.0   # -6° to 0°
LON_MIN_E, LON_MAX_E = 0.0,   42.0    #  0° to 42°

sst = ds["sea_surface_temperature"]

# Subset latitude (lat is stored descending: 90 → -90)
sst_lat = sst.sel(lat=slice(LAT_MAX, LAT_MIN))

# Combine western and eastern longitude slices, then sort by lon
sst_west = sst_lat.sel(lon=slice(LON_MIN_W, LON_MAX_W))
sst_east = sst_lat.sel(lon=slice(LON_MIN_E, LON_MAX_E))
sst_med  = xr.concat([sst_west, sst_east], dim="lon").sortby("lon")

# Convert longitude from 0–360 to -180–180 for a nicer map
# Correct formula: (lon + 180) % 360 - 180  maps 354→-6, 360/0→0, 42→42
sst_med = sst_med.assign_coords(lon=((sst_med.lon.values + 180) % 360 - 180))
sst_med = sst_med.sortby("lon")

print("Mediterranean SST shape:", sst_med.dims)
sst_med

In [ ]:
# Compute monthly mean and convert from Kelvin to Celsius
sst_mean_K = sst_med.mean(dim="time").compute()
sst_mean_C = sst_mean_K - 273.15
sst_mean_C.attrs["units"] = "°C"
sst_mean_C.attrs["long_name"] = "Sea Surface Temperature"

month_label = np.datetime_as_string(sst_med.time.values[0], unit="M")
print(f"Monthly mean SST computed for {month_label}")
print(f"  Min: {float(sst_mean_C.min()):.1f} °C   Max: {float(sst_mean_C.max()):.1f} °C")

In [ ]:
import holoviews as hv

# Fix uneven-sampling warning: after coord reassignment lon spacing has
# tiny floating-point jitter; relax the tolerance so hvplot uses Image correctly
hv.config.image_rtol = 0.01

# Compute tight color limits from the actual data (land is NaN, ignored)
vmin = float(sst_mean_C.min())
vmax = float(sst_mean_C.max())

sst_plot = sst_mean_C.hvplot(
    x="lon",
    y="lat",
    geo=True,
    rasterize=True,
    cmap="inferno",        # built-in matplotlib colormap, works without extra packages
    clim=(vmin, vmax),
    clabel="SST (°C)",
    alpha=0.85,
    coastline=True,
    tiles="EsriImagery",
    frame_width=800,
    frame_height=450,
    xlim=(-8, 44),
    ylim=(28, 49),
    title=f"ERA5 Mean Sea Surface Temperature — Mediterranean ({month_label})",
    fontscale=1.2,
)

sst_plot